# 必勝賽馬方程式? 機器學習的預測分析應用

本文示範如何使用 Python 重現 `HorseraceExample.R` 中的分析流程。資料來自香港賽馬會，步驟涵蓋特徵工程、建模及評估。

## 載入套件
這裡使用 `pandas`、`numpy`、`matplotlib` 等常見套件，`xgboost` 及 `scikit-learn` 來訓練模型。

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVR
import xgboost as xgb

## 讀取資料
示例檔案為 `Database 2008-2009.xlsx`。

In [ ]:
# 載入 Excel 資料
df = pd.read_excel('Database 2008-2009.xlsx')
# 依照 Run 排序
df = df.sort_values('Run').reset_index(drop=True)

## 特徵工程
以下程式碼根據 R 範例計算各種特徵，包括上場表現、移動平均等。

In [ ]:
# 班次轉換
class_map = {7:1, 1:2, 2:3, 3:4, 4:5, 5:6, 6:7}
df['Class.Cal'] = df['Class'].map(class_map)
# 上場班次差異
df['LagClass'] = df.groupby('Name')['Class.Cal'].shift(1)
df['L1Class'] = np.where(df['LagClass'].isna(), 0, df['Class.Cal'] - df['LagClass'])
# 上場名次
df['L1FinPos'] = df.groupby('Name')['FinPos'].shift(1)
avg_fp = df['L1FinPos'].mean()
df['AVG4FinPos'] = df.groupby('Name')['L1FinPos'].apply(lambda x: x.rolling(4, min_periods=1).mean()).fillna(avg_fp)
# 評分及體重差異
df['L1Rating'] = df.groupby('Name')['Rating'].diff().fillna(0)
df['L1HrWt'] = df.groupby('Name')['HrWt'].diff().fillna(0)
# 上場距今天數
df['date1'] = pd.to_datetime(df['Date'])
df['date2'] = df.groupby('Name')['date1'].shift(1)
df['date3'] = (df['date1'] - df['date2']).dt.days.fillna(365)
df['LastRun'] = np.where(df['date3']>365, 365, df['date3'])
# 勝出與否
df['FO'] = (df['FinPos']==1).astype(int)

## 分割訓練與測試資料

In [ ]:
train = df[df['Date'] <= '2009-12-31']
test = df[df['Date'] > '2009-12-31']

## 建立模型
此處示範邏輯迴歸、XGBoost、隨機森林及支持向量機。

In [ ]:
features = ['Age','HrWt','WtCr','L1Class','AVG4FinPos','L1Rating','L1HrWt','LastRun']
X_train = train[features]
X_test = test[features]
y_train = train['FO']

# Logistic Regression
logit = LogisticRegression(max_iter=1000)
logit.fit(X_train, y_train)
pred_logit = logit.predict_proba(X_test)[:,1]

# XGBoost
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test)
params = {'objective':'binary:logistic'}
bst = xgb.train(params, dtrain, num_boost_round=1)
pred_xgb = bst.predict(dtest)

# Random Forest
rf = RandomForestClassifier(n_estimators=100)
rf.fit(X_train, y_train)
pred_rf = rf.predict_proba(X_test)[:,1]

# Support Vector Machine (回歸模式)
svm = SVR()
svm.fit(train[features], train['FinPos'])
pred_svm = svm.predict(test[features])

## 評估與排序
依照各模型預測機率或名次排序，與實際結果比較。

In [ ]:
result = test.copy()
result['pred_logit'] = pred_logit
result['pred_xgb'] = pred_xgb
result['pred_rf'] = pred_rf
result['pred_svm'] = pred_svm
# 以賠率排序的熱門馬
result['rankO'] = result.groupby('Run')['FinOdd'].rank(method='min')
result['rank_logit'] = result.groupby('Run')['pred_logit'].rank(ascending=False, method='min')
result['rank_xgb'] = result.groupby('Run')['pred_xgb'].rank(ascending=False, method='min')
result['rank_rf'] = result.groupby('Run')['pred_rf'].rank(ascending=False, method='min')
result['rank_svm'] = result.groupby('Run')['pred_svm'].rank(method='min')

## 小結
本 Notebook 以 Python 重現原始 R 程式的主要流程，包含資料處理、特徵工程及多種模型訓練。更完整的說明請參考原始 R 版說明文件。